# AI Production Studio — casting de personnages FR / AR

**16 profils cibles, à créer et écouter** : timbres féminins/masculins × petite enfance (~5 ans), enfant (~10), adolescent (~15), jeunes adultes (~20), adultes (~30, ~40, ~50), seniors (~70). Chaque voix peut être auditionnée en français et en arabe : 32 combinaisons cibles, pas 32 voix déjà validées. Plusieurs variantes sont possibles dans chaque profil. Âges et genres désignent des intentions de timbre, pas l’âge ou le genre d’une personne réelle.

**Méthode expérimentale** : Qwen VoiceDesign crée une référence fictive en français à partir d’une description. Chatterbox réutilise exactement ce fichier en FR ou AR. Qwen ne prend pas directement en charge l’arabe. Aucune prise de voix réelle ou photo n’est nécessaire. L’identité peut varier pendant le transfert, et une référence française peut influencer l’accent arabe : il faut approuver naturel et diction dans chaque langue.

**Utilisation**
1. Gardez votre session L4 existante. Exécutez l’installation et la préparation.
2. Choisissez PROFIL et VARIANTE pour créer **une seule référence**. Écoutez-la ; changez la variante pour auditionner un autre candidat.
3. Testez une réplique en AR puis FR, en conservant la référence. Changez seulement LANGUE pour le second essai.
4. Le dossier Drive conserve automatiquement la voix et son JSON avec votre personnage. Aux prochaines sessions, choisissez « Réutiliser une voix sauvegardée » : la même référence, son empreinte et la révision Chatterbox seront reprises. Ne régénérez pas le timbre de zéro à chaque réplique.
5. Importez les WAV de répliques dans le studio avec **Importer un média**. Le studio ne dispose pas encore d’un sélecteur de casting connecté à ces notebooks.

Deux environnements isolés libèrent la mémoire du premier moteur quand son processus se termine, avant de lancer le second. Le notebook n’exécute pas tous les profils en série. Les poids et dépendances représentent plusieurs Go et le premier chargement peut être long. Aucun achat ni API payante ; votre session Colab continue de consommer ses unités. Aucun serveur ni tunnel public n’est ouvert.

Conservez la référence originale et la provenance JSON. La même seed seule ne garantit pas une identité stable ; réutiliser la référence aide mais ne garantit pas un résultat identique entre langues et textes. La durée de référence doit rester entre 3 et 20 secondes. Le texte arabe vocalisé est transmis sans vocalisation automatique ; les harakat, pauses et mots omis/répétés restent à écouter.

Sources : [Qwen3-TTS officiel](https://github.com/QwenLM/Qwen3-TTS), modèle [VoiceDesign](https://huggingface.co/Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign) publié sous Apache 2.0 ; [Chatterbox](https://github.com/resemble-ai/chatterbox) publié sous MIT, filigrane natif conservé. Seuls les fichiers safetensors/configuration sont téléchargés pour Qwen. Révisions figées et empreintes observées conservées.

**État de validation** : le propriétaire a apprécié les auditions Kokoro FR et Chatterbox AR de base. Installation/import Qwen et Chatterbox vérifiés ici, ainsi que conservation/contrôle des références. La création Qwen sur L4 et le transfert des 16 profils ne sont pas encore validés : le premier profil constitue un essai, pas une promesse de catalogue complet.

## Rangement automatique des voix

Au premier lancement, Google Colab peut demander la connexion à Drive. Les créations sont enregistrées dans **Mon Drive / AI Production Studio / Voix**, dans un dossier par profil/variante/référence (ex. `M-petit_var1_ab12cd34ef56`). Chaque référence et chaque réplique ont leur **WAV et JSON portant le même nom**. AR et FR restent dans ce dossier ; les nouvelles prises sont numérotées sans écraser les précédentes. Le sous-ZIP `voix-fictive.zip` conserve la référence réutilisable. Le suffixe de référence distingue deux créations différentes portant le même profil et numéro de variante.

Par défaut, aucun téléchargement manuel n’est nécessaire : le dossier Drive et son catalogue sont mis à jour après chaque création et chaque réplique. La case TELECHARGER_AUTOMATIQUEMENT permet, si vous le souhaitez, de recevoir aussi un ZIP sur le PC. Chrome peut demander d’autoriser les téléchargements multiples pour Colab. Il le place dans son dossier de téléchargement configuré ; un notebook ne peut pas choisir librement un dossier Windows ni extraire un ZIP sur le PC. Vous pouvez choisir un dossier de téléchargement dans les paramètres Chrome. Si vous désactivez Drive, la copie Colab locale est temporaire : conservez les ZIP téléchargés. Le notebook n’importe pas automatiquement ces voix dans le studio Windows.

Le catalogue `catalogue_voix.json` du dossier Drive indique les profils, variantes, références et fichiers sauvegardés. Il n’est pas transmis automatiquement à cette conversation. Les boutons de création restent sous votre contrôle ; le notebook n’exécute pas de lots ou de dépenses sans que vous lanciez les cellules.


In [ ]:
# Une connexion Google peut être demandée pour la copie persistante dans votre Drive.
SAUVEGARDER_DANS_DRIVE = True #@param {type:"boolean"}
TELECHARGER_AUTOMATIQUEMENT = False #@param {type:"boolean"}
from pathlib import Path
if SAUVEGARDER_DANS_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    VOICES_ROOT = Path('/content/drive/MyDrive/AI Production Studio/Voix')
else:
    VOICES_ROOT = Path('/content/AI_Production_Studio_Voix')
VOICES_ROOT.mkdir(parents=True, exist_ok=True)
print('Dossier des voix :', VOICES_ROOT)

REQUIREMENTS = '''
# Python 3.11, Linux x86_64 ; installation/import vérifiés le 22 octobre 2026.
aiofiles==24.1.0
annotated-doc==0.0.5
annotated-types==0.8.0
anyio==4.15.1
audioread==3.1.0
brotli==1.2.0
catalogue==2.0.10
certifi==2026.10.19
cffi==2.1.1
cfgv==3.5.0
charset-normalizer==3.5.2
chatterbox-tts==0.1.7
click==8.5.0
cloudpickle==3.1.2
conformer==0.3.2
decorator==5.3.1
deprecated==1.3.1
diffusers==0.29.0
distlib==0.4.4
einops==0.8.2
fastapi==0.159.1
ffmpy==1.0.0
filelock==4.2.4
fsspec==2026.9.0
gradio==6.8.0
gradio-client==2.2.0
groovy==0.1.2
h11==0.16.0
hf-xet==1.7.0
httpcore==1.0.9
httpx==0.28.1
huggingface-hub==1.33.0
identify==2.6.20
idna==3.20
importlib-metadata==9.0.1
jaconv==0.5.0
jinja2==3.1.6
joblib==1.6.0
lazy-loader==0.6
librosa==0.11.0
llvmlite==0.50.0
markdown-it-py==4.2.0
markupsafe==3.0.4
mdurl==0.1.2
ml-dtypes==0.5.4
mpmath==1.3.0
msgpack==1.2.3
narwhals==2.27.1
networkx==3.6.1
nodeenv==1.11.0
numba==0.68.0
numpy==1.26.4
nvidia-cublas-cu12==12.4.5.8
nvidia-cuda-cupti-cu12==12.4.127
nvidia-cuda-nvrtc-cu12==12.4.127
nvidia-cuda-runtime-cu12==12.4.127
nvidia-cudnn-cu12==9.1.0.70
nvidia-cufft-cu12==11.2.1.3
nvidia-curand-cu12==10.3.5.147
nvidia-cusolver-cu12==11.6.1.9
nvidia-cusparse-cu12==12.3.1.170
nvidia-cusparselt-cu12==0.6.2
nvidia-nccl-cu12==2.21.5
nvidia-nvjitlink-cu12==12.4.127
nvidia-nvtx-cu12==12.4.127
omegaconf==2.4.0
onnx==1.23.2
opentelemetry-api==1.45.1
orjson==3.13.0
packaging==26.3
pandas==3.0.6
pillow==12.3.0
platformdirs==4.13.5
pooch==1.9.0
pre-commit==4.7.0
protobuf==7.36.2
pycparser==3.11
pydantic==2.14.1
pydantic-core==2.50.1
pydub==0.25.1
pygments==2.21.0
pykakasi==2.3.0
pyloudnorm==0.2.0
python-dateutil==3.0.0
python-discovery==1.6.4
python-multipart==0.0.32
pytz==2026.5
pyyaml==6.0.3
regex==2026.9.29
requests==2.34.2
resemble-perth==1.0.1
rich==15.0.0
s3tokenizer==0.3.0
safehttpx==0.1.7
safetensors==0.5.3
scikit-learn==1.9.1
scipy==1.17.1
semantic-version==2.10.0
setuptools==80.9.0
shellingham==1.5.4
six==1.17.0
soundfile==0.13.1
soxr==1.1.0
spacy-pkuseg==1.0.1
srsly==2.5.4
starlette==0.52.1
sympy==1.13.1
threadpoolctl==3.7.0
tokenizers==0.22.2
tomlkit==0.13.3
torch==2.6.0
torchaudio==2.6.0
tqdm==4.70.1
transformers==5.2.0
triton==3.2.0
typer==0.27.3
typer-slim==0.24.0
typing-extensions==4.16.0
typing-inspection==0.4.4
tzdata==2026.5
urllib3==2.8.0
uvicorn==0.54.0
virtualenv==21.14.12
wrapt==2.5.1
zipp==4.1.2
'''
# Une installation indépendante : le Python et le PyTorch du notebook restent en place.
import os, subprocess, sys
from pathlib import Path
subprocess.check_call([sys.executable, "-m", "pip", "install", "uv==0.12.23"])
runtime = Path(".studio-chatterbox-runtime").resolve()
runtime.mkdir(exist_ok=True)
worker_env = os.environ.copy()
worker_env["UV_CACHE_DIR"] = str(runtime / "uv-cache")
worker_env["UV_PYTHON_INSTALL_DIR"] = str(runtime / "python")
worker_env["HF_HOME"] = str(runtime / "hf-cache")
uv_command = [sys.executable, "-m", "uv", "--system-certs"]
venv = runtime / "venv"
worker_python = venv / "bin" / "python"
if not worker_python.exists():
    subprocess.check_call(uv_command + ["venv", "--python", "3.11.17", str(venv)], env=worker_env)
lock_path = runtime / "requirements.txt"
lock_path.write_text(REQUIREMENTS, encoding="utf-8")
subprocess.check_call(uv_command + ["pip", "install", "--python", str(worker_python), "-r", str(lock_path)], env=worker_env)
preflight = subprocess.run([str(worker_python), "-c", "from chatterbox.mtl_tts import ChatterboxMultilingualTTS; import torch, perth; assert callable(perth.PerthImplicitWatermarker), 'Perth indisponible'; marker = perth.PerthImplicitWatermarker(); print('Perth OK. Torch isolé :', torch.__version__); print('GPU disponible :', torch.cuda.is_available())"], env=worker_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(preflight.stdout, flush=True)
preflight.check_returncode()
print("Environnement prêt. Cette cellule n'a pas lancé de synthèse.")

QWEN_REQUIREMENTS = '# Python 3.11.17 ; installation et import vérifiés, génération GPU à tester.\naccelerate==1.12.0\nannotated-doc==0.0.5\nannotated-types==0.8.0\nanyio==4.15.1\naudioread==3.1.0\nbrotli==1.2.0\ncertifi==2026.10.19\ncffi==2.1.1\ncharset-normalizer==3.5.2\nclick==8.5.0\ncloudpickle==3.1.2\ndecorator==5.3.1\neinops==0.8.2\nfastapi==0.159.1\nfilelock==4.2.4\nflatbuffers==25.12.19\nfsspec==2026.9.0\ngradio==6.17.3\ngradio-client==2.5.0\ngroovy==0.1.2\nh11==0.16.0\nhf-gradio==0.4.1\nhf-xet==1.7.0\nhttpcore==1.0.9\nhttpx==0.28.1\nhuggingface-hub==0.36.2\nidna==3.20\njinja2==3.1.6\njoblib==1.6.0\nlazy-loader==0.6\nlibrosa==0.11.0\nllvmlite==0.50.0\nmarkdown-it-py==4.2.0\nmarkupsafe==3.0.4\nmdurl==0.1.2\nmpmath==1.3.0\nmsgpack==1.2.3\nnarwhals==2.27.1\nnetworkx==3.6.1\nnumba==0.68.0\nnumpy==2.4.6\nnvidia-cublas-cu12==12.4.5.8\nnvidia-cuda-cupti-cu12==12.4.127\nnvidia-cuda-nvrtc-cu12==12.4.127\nnvidia-cuda-runtime-cu12==12.4.127\nnvidia-cudnn-cu12==9.1.0.70\nnvidia-cufft-cu12==11.2.1.3\nnvidia-curand-cu12==10.3.5.147\nnvidia-cusolver-cu12==11.6.1.9\nnvidia-cusparse-cu12==12.3.1.170\nnvidia-cusparselt-cu12==0.6.2\nnvidia-nccl-cu12==2.21.5\nnvidia-nvjitlink-cu12==12.4.127\nnvidia-nvtx-cu12==12.4.127\nonnxruntime==1.31.0\nopentelemetry-api==1.45.1\norjson==3.13.0\npackaging==26.3\npandas==3.0.6\npillow==12.3.0\nplatformdirs==4.13.5\npooch==1.9.0\nprotobuf==7.36.2\npsutil==7.2.2\npycparser==3.11\npydantic==2.14.1\npydantic-core==2.50.1\npydub==0.25.1\npygments==2.21.0\npython-dateutil==3.0.0\npython-multipart==0.0.32\npytz==2026.5\npyyaml==6.0.3\nqwen-tts==0.1.1\nregex==2026.9.29\nrequests==2.34.2\nrich==15.0.0\nsafehttpx==0.1.7\nsafetensors==0.8.0\nscikit-learn==1.9.1\nscipy==1.17.1\nsemantic-version==2.10.0\nshellingham==1.5.4\nsix==1.17.0\nsoundfile==0.13.1\nsox==1.5.0\nsoxr==1.1.0\nstarlette==1.9.0\nsympy==1.13.1\nthreadpoolctl==3.7.0\ntokenizers==0.22.2\ntomlkit==0.14.0\ntorch==2.6.0\ntorchaudio==2.6.0\ntqdm==4.70.1\ntransformers==4.57.3\ntriton==3.2.0\ntyper==0.27.3\ntyping-extensions==4.16.0\ntyping-inspection==0.4.4\ntzdata==2026.5\nurllib3==2.8.0\nuvicorn==0.54.0\n'
qwen_python = runtime / 'qwen-venv' / 'bin' / 'python'
if not qwen_python.exists():
    subprocess.check_call(uv_command + ['venv', '--python', '3.11.17', str(runtime / 'qwen-venv')], env=worker_env)
qwen_lock = runtime / 'qwen-requirements.txt'
qwen_lock.write_text(QWEN_REQUIREMENTS, encoding='utf-8')
subprocess.check_call(uv_command + ['pip', 'install', '--python', str(qwen_python), '-r', str(qwen_lock)], env=worker_env)
probe = subprocess.run([str(qwen_python), '-c', 'from qwen_tts import Qwen3TTSModel; print("Qwen VoiceDesign : import OK")'], env=worker_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(probe.stdout, flush=True)
probe.check_returncode()


In [ ]:
QWEN_SOURCE = r'''
"""Création de références fictives par description ; un candidat à la fois."""
from pathlib import Path
import argparse
import hashlib
import json
import re
import time
import zipfile

MODEL_ID = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
REFERENCE_TEXT = "Bonjour. Une nouvelle histoire commence. Aujourd’hui, nous allons découvrir un endroit merveilleux, et partager cette aventure ensemble."
AGE_GROUPS = [
    ("petit", "Petite enfance · environ 5 ans", "a young child, around five years old, with light natural articulation"),
    ("enfant", "Enfant · environ 10 ans", "a child, around ten years old, with clear lively articulation"),
    ("ado", "Adolescent · environ 15 ans", "a teenager, around fifteen years old, with a natural youthful timbre"),
    ("20", "Jeune adulte · environ 20 ans", "a young adult, around twenty years old"),
    ("30", "Adulte · environ 30 ans", "an adult, around thirty years old"),
    ("40", "Adulte · environ 40 ans", "an adult, around forty years old"),
    ("50", "Adulte · environ 50 ans", "an adult, around fifty years old"),
    ("senior", "Senior · environ 70 ans", "an elderly adult, around seventy years old, with a gently aged timbre and clear diction"),
]


def profiles():
    return [{"id": sex+"-"+key, "label": label+" · "+age,
             "age_target": age, "gender_target": label, "languages_to_audition": ["fr", "ar"],
             "status": "À créer et auditionner", "reference_sha256": None,
             "instruction": "A fictional "+gender+" voice of "+description+
             ". Speak French naturally, with clear pronunciation, a calm friendly tone, and normal conversational pace. Dry close-microphone recording, no music, no background noise. Avoid caricature or exaggerated pitch."}
            for sex, label, gender in [("F", "Féminine", "female"), ("M", "Masculine", "male")]
            for key, age, description in AGE_GROUPS]


def digest(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def design(profile_id, directory, variant=1):
    candidates = {p["id"]: p for p in profiles()}
    if profile_id not in candidates or isinstance(variant, bool) or not isinstance(variant, int) or not 1 <= variant <= 99:
        raise ValueError("Choisissez un profil du casting et une variante entre 1 et 99.")
    import numpy as np
    import torch
    import soundfile as sf
    from qwen_tts import Qwen3TTSModel
    from huggingface_hub import model_info, snapshot_download
    if not torch.cuda.is_available():
        raise RuntimeError("Choisissez votre session L4 existante pour cet essai.")
    free, _ = torch.cuda.mem_get_info()
    if free < 10 * 1024**3:
        raise RuntimeError("Moins de 10 Go de VRAM libres. Arrêtez les autres calculs avant cet essai.")
    folder = Path(directory)
    folder.mkdir(parents=True, exist_ok=False)
    seed = 4200 + variant
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    started = time.perf_counter()
    revision = model_info(MODEL_ID, token=False).sha
    if not isinstance(revision, str) or not re.fullmatch(r"[0-9a-f]{40}", revision):
        raise RuntimeError("Révision officielle Qwen introuvable.")
    checkpoint = Path(snapshot_download(MODEL_ID, revision=revision, token=False,
        allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.tiktoken"]))
    model = Qwen3TTSModel.from_pretrained(str(checkpoint), device_map="cuda:0",
        dtype=torch.bfloat16, attn_implementation="sdpa", use_safetensors=True,
        local_files_only=True)
    profile = candidates[profile_id]
    print("Création d'un candidat :", profile["label"], "variante", variant, flush=True)
    with torch.inference_mode():
        waves, rate = model.generate_voice_design(text=REFERENCE_TEXT, language="French",
            instruct=profile["instruction"], max_new_tokens=256)
    torch.cuda.synchronize()
    audio = np.asarray(waves[0], dtype=np.float32).squeeze()
    if audio.ndim != 1 or not audio.size or not np.isfinite(audio).all() or not np.max(np.abs(audio)):
        raise RuntimeError("La référence produite est vide ou invalide.")
    duration = len(audio) / rate
    if not 3 <= duration <= 20:
        raise RuntimeError("Durée de référence hors plage 3–20 s : " + str(round(duration, 1)))
    reference = folder / "reference.wav"
    sf.write(str(reference), audio, rate, subtype="PCM_16")
    report = {"schema_version": 1, "source_type": "fictional_voice_design",
        "profile_id": profile_id, "label": profile["label"], "variant": variant,
        "age_target": profile["age_target"], "gender_target": profile["gender_target"],
        "instruction": profile["instruction"], "reference_text": REFERENCE_TEXT,
        "reference_language": "fr", "reference_sha256": digest(reference),
        "reference_duration_sec": duration, "model_id": MODEL_ID, "model_revision": revision,
        "seed": seed, "dtype": "bfloat16", "attention": "sdpa", "max_new_tokens": 256,
        "total_sec_including_download": round(time.perf_counter()-started, 3),
        "audition_status": {"reference": "À écouter", "fr": "Non auditionnée", "ar": "Non auditionnée"},
        "limits": "Âge/timbre demandés, pas garantis. Identité, accent et diction à approuver dans chaque langue.",
        "model_sha256_observed": {str(p.relative_to(checkpoint)): digest(p) for p in checkpoint.rglob("*.safetensors")}}
    manifest = folder / "voix.json"
    manifest.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    archive = folder / "voix-fictive.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(reference, "reference.wav")
        z.write(manifest, "voix.json")
    print("Référence à écouter :", reference, flush=True)
    return reference


def restore_reference(archive, directory):
    """Restaure seulement une référence de ce casting, sans extraire de chemins fournis."""
    archive = Path(archive)
    if archive.stat().st_size > 12 * 1024**2:
        raise ValueError("Archive de voix : 12 Mo maximum.")
    with zipfile.ZipFile(archive) as z:
        entries = z.infolist()
        if len(entries) != 2 or set(z.namelist()) != {"voix.json", "reference.wav"}:
            raise ValueError("Archive de référence invalide.")
        if sum(e.file_size for e in entries) > 12 * 1024**2 or z.getinfo("voix.json").file_size > 100000:
            raise ValueError("Archive de voix trop volumineuse.")
        report = json.loads(z.read("voix.json"))
        audio = z.read("reference.wav")
    if report.get("schema_version") != 1 or report.get("source_type") != "fictional_voice_design":
        raise ValueError("Ce casting attend une référence fictive créée par le notebook.")
    if hashlib.sha256(audio).hexdigest() != report.get("reference_sha256"):
        raise ValueError("Empreinte de la référence incorrecte.")
    if not (audio.startswith(b"RIFF") and audio[8:12] == b"WAVE"):
        raise ValueError("La référence n'est pas un fichier WAV.")
    folder = Path(directory)
    folder.mkdir(parents=True, exist_ok=False)
    reference = folder / "reference.wav"
    reference.write_bytes(audio)
    (folder / "voix.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return reference


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--profile", choices=[p["id"] for p in profiles()], required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--variant", type=int, default=1)
    args = parser.parse_args()
    design(args.profile, args.output_dir, args.variant)
'''
CHATTERBOX_SOURCE = r'''
"""Audition Chatterbox FR/AR ; voix par défaut ou référence fictive conservée."""
from pathlib import Path
import argparse
import hashlib
import json
import random
import re
import time

MODEL_ID = "ResembleAI/chatterbox"
MODEL_FILES = [
    "ve.pt", "t3_mtl23ls_v2.safetensors", "s3gen.pt",
    "grapheme_mtl_merged_expanded_v1.json", "conds.pt", "Cangjie5_TC.json",
]
DEFAULT_TEXT = "مَرْحَبًا بِكُمْ فِي وَرْشَتِنَا. عَلِمَ الطَّالِبُ. عُلِمَ الْخَبَرُ. دَرَسَ الطَّالِبُ. دَرَّسَ الْمُعَلِّمُ."


def validate_text(text, language="ar"):
    if language not in ("ar", "fr"):
        raise ValueError("Langue : ar ou fr.")
    if not isinstance(text, str) or not 1 <= len(text.strip()) <= 500:
        raise ValueError("Essai : un texte de 1 à 500 caractères.")
    if language == "ar" and (not re.search(r"[\u0621-\u064a]", text) or not re.search(r"[\u064b-\u0652]", text)):
        raise ValueError("Utilisez un texte arabe vocalisé pour comparer la diction.")
    return text.strip()


def validate_reference(path, expected_sha256):
    import soundfile as sf
    path = Path(path)
    if not path.is_file() or path.stat().st_size > 10 * 1024**2:
        raise ValueError("Référence WAV absente ou supérieure à 10 Mo.")
    if not re.fullmatch(r"[0-9a-f]{64}", expected_sha256 or "") or digest(path) != expected_sha256:
        raise ValueError("La référence vocale ne correspond pas à l'empreinte conservée.")
    info = sf.info(path)
    if info.format != "WAV" or not 3 <= info.duration <= 20 or info.channels not in (1, 2):
        raise ValueError("Référence : WAV de 3 à 20 secondes, mono ou stéréo.")
    return path


def phoneme_checks(tokenizer):
    """Vérifie la distinction dans les tokens, sans prétendre valider la diction."""
    checks = []
    for left, right in [("عَلِمَ", "عُلِمَ"), ("دَرَسَ", "دَرَّسَ"), ("قَلَمٌ", "قَلَمٍ")]:
        a = tokenizer.text_to_tokens(left, language_id="ar").tolist()
        b = tokenizer.text_to_tokens(right, language_id="ar").tolist()
        checks.append({"left": left, "right": right, "distinct_tokens": a != b})
    return checks


def digest(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def generate(text, directory, seed=42, device="cuda", language="ar", reference=None,
             reference_sha256=None, voice_name=None, model_revision=None):
    text = validate_text(text, language)
    if reference is not None:
        reference = validate_reference(reference, reference_sha256)
    if device not in ("cuda", "cpu"):
        raise ValueError("Périphérique non pris en charge.")
    import numpy as np
    import torch
    import soundfile as sf
    import perth
    if not callable(perth.PerthImplicitWatermarker):
        raise RuntimeError("Perth incomplet. Installez setuptools==80.9.0 dans l'environnement isolé, puis relancez ce programme.")
    from huggingface_hub import model_info, snapshot_download
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS
    from importlib.metadata import version

    if version("chatterbox-tts") != "0.1.7":
        raise RuntimeError("Ce notebook nécessite chatterbox-tts 0.1.7 dans son environnement isolé.")
    if device == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("GPU absent. Sélectionnez votre session L4 existante, sans souscrire d'offre supplémentaire.")
        free, _ = torch.cuda.mem_get_info()
        if free < 6 * 1024**3:
            raise RuntimeError("Moins de 6 Go de VRAM libres. Arrêtez les anciens calculs avant l'essai.")
    folder = Path(directory)
    folder.mkdir(parents=True, exist_ok=False)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if device == "cuda":
        torch.cuda.manual_seed_all(seed)
    started = time.perf_counter()
    revision = model_revision or model_info(MODEL_ID, token=False).sha
    if not isinstance(revision, str) or not re.fullmatch(r"[0-9a-f]{40}", revision):
        raise RuntimeError("Révision officielle du modèle introuvable.")
    print("Téléchargement du modèle officiel, révision :", revision, flush=True)
    checkpoint = Path(snapshot_download(repo_id=MODEL_ID, revision=revision,
                                        allow_patterns=MODEL_FILES, token=False))
    required = set(MODEL_FILES) - {"Cangjie5_TC.json"}
    if any(not (checkpoint / name).is_file() for name in required):
        raise RuntimeError("Fichiers du modèle incomplets, y compris la voix fournie par le modèle.")
    print("Chargement de Chatterbox Multilingual V2...", flush=True)
    model = ChatterboxMultilingualTTS.from_local(checkpoint, device=device)
    checks = phoneme_checks(model.tokenizer) if language == "ar" else []
    if not all(check["distinct_tokens"] for check in checks):
        raise RuntimeError("Ce tokenizer ne distingue pas tous les signes testés. Ne pas retenir cet essai pour l'arabe vocalisé.")
    loaded = time.perf_counter()
    print("Synthèse", language, "avec référence fictive conservée." if reference else "avec la voix par défaut.", flush=True)
    # cfg_weight=0 suit la recommandation du mainteneur pour limiter le transfert
    # d'accent depuis une voix de référence d'une autre langue. Résultat à écouter.
    cfg_weight = 0.3 if reference is not None else 0.0
    kwargs = {"audio_prompt_path": str(reference)} if reference is not None else {}
    with torch.inference_mode():
        waveform = model.generate(text, language_id=language, exaggeration=0.5,
                                  cfg_weight=cfg_weight, temperature=0.8, **kwargs)
    if device == "cuda":
        torch.cuda.synchronize()
    finished = time.perf_counter()
    samples = waveform.detach().cpu().float().numpy().squeeze()
    if samples.ndim != 1 or not samples.size or not np.isfinite(samples).all() or not np.max(np.abs(samples)):
        raise RuntimeError("Résultat audio vide ou invalide.")
    output = folder / ("essai-chatterbox-" + language + ".wav")
    sf.write(str(output), samples, model.sr, subtype="PCM_16")
    report = {
        "model_id": MODEL_ID, "model_revision": revision, "variant": "Multilingual V2",
        "engine": "chatterbox-tts 0.1.7", "torch_version": torch.__version__,
        "device": device, "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
        "language": language, "text": text, "automatic_vowelizer": None,
        "voice": voice_name or "Voix fournie avec le modèle",
        "reference_sha256": reference_sha256 if reference is not None else None,
        "reference_language": "fr" if reference is not None else None,
        "cross_language_identity": "Identité et accent à approuver séparément en FR et AR.",
        "seed": seed, "exaggeration": 0.5, "cfg_weight": cfg_weight, "temperature": 0.8,
        "sample_rate": model.sr, "duration_sec": len(samples) / model.sr,
        "load_sec_including_download": round(loaded-started, 3),
        "generation_sec": round(finished-loaded, 3), "token_checks": checks,
        "model_sha256_observed": {name: digest(checkpoint / name) for name in sorted(required)},
        "audio_sha256": digest(output), "watermark": "Filigrane natif Chatterbox conservé.",
        "quality": "Naturel, accent, harakat, mots omis ou répétés à vérifier à l'écoute. Aucune diction parfaite garantie.",
    }
    output.with_suffix(".json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Audio produit :", output, flush=True)
    return output


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--text-file", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--device", choices=["cuda", "cpu"], default="cuda")
    parser.add_argument("--language", choices=["ar", "fr"], default="ar")
    parser.add_argument("--reference", type=Path)
    parser.add_argument("--reference-sha256")
    parser.add_argument("--voice-name")
    parser.add_argument("--model-revision")
    args = parser.parse_args()
    generate(args.text_file.read_text(encoding="utf-8"), args.output_dir, device=args.device,
             language=args.language, reference=args.reference, reference_sha256=args.reference_sha256,
             voice_name=args.voice_name, model_revision=args.model_revision)
'''
import runpy, zipfile, json, hashlib
qwen_script = runtime / 'casting_voix.py'
chat_script = runtime / 'casting_chatterbox.py'
qwen_script.write_text(QWEN_SOURCE, encoding='utf-8')
chat_script.write_text(CHATTERBOX_SOURCE, encoding='utf-8')
casting = runpy.run_path(str(qwen_script))
profiles = {p['id']: p for p in casting['profiles']()}
def run_visible(command):
    with subprocess.Popen(command, env=worker_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
    if status:
        raise RuntimeError('Le moteur s’est arrêté ; voir son diagnostic juste au-dessus. Code : '+str(status))
print('16 profils cibles. Un profil ne devient une voix disponible qu’après génération et écoute.')

ARCHIVE_SOURCE = r'''
"""Rangement des WAV et JSON par profil/variante, indépendant des moteurs."""
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import shutil
import uuid
import zipfile


def digest(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def save_voice(root, reference_dir, audition=None):
    root, reference_dir = Path(root), Path(reference_dir)
    voice = json.loads((reference_dir / "voix.json").read_text(encoding="utf-8"))
    profile, variant = voice.get("profile_id"), voice.get("variant")
    if not isinstance(profile, str) or not re.fullmatch(r"[FM]-(?:petit|enfant|ado|20|30|40|50|senior)", profile):
        raise ValueError("Profil vocal invalide.")
    if isinstance(variant, bool) or not isinstance(variant, int) or not 1 <= variant <= 99:
        raise ValueError("Variante invalide.")
    reference = reference_dir / "reference.wav"
    sha = digest(reference)
    if sha != voice.get("reference_sha256"):
        raise ValueError("Référence modifiée : sauvegarde arrêtée.")
    report = None
    if audition is not None:
        audition = Path(audition)
        report = json.loads(audition.with_suffix(".json").read_text(encoding="utf-8"))
        if report.get("reference_sha256") != sha or report.get("language") not in ("ar", "fr"):
            raise ValueError("La réplique ne correspond pas à cette voix.")
        if digest(audition) != report.get("audio_sha256"):
            raise ValueError("Réplique modifiée : sauvegarde arrêtée.")
    prefix = profile + "_var" + str(variant)
    folder = root / (prefix + "_" + sha[:12])
    folder.mkdir(parents=True, exist_ok=True)
    saved_reference = folder / (prefix + "_reference.wav")
    if saved_reference.exists():
        if digest(saved_reference) != sha:
            raise ValueError("Une référence différente existe déjà dans ce dossier.")
    else:
        shutil.copy2(reference, saved_reference)
    # Cette fiche reflète la dernière révision/les auditions ; les WAV restent immuables.
    (folder / (prefix + "_reference.json")).write_text(json.dumps(voice, ensure_ascii=False, indent=2)+"\n", encoding="utf-8")
    with zipfile.ZipFile(folder / "voix-fictive.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(saved_reference, "reference.wav")
        z.writestr("voix.json", json.dumps(voice, ensure_ascii=False, indent=2))
    saved_audition = None
    if report is not None:
        base = prefix + "_" + report["language"].upper()
        take = 1
        while True:
            destination = folder / (base + "_prise" + str(take).zfill(2) + ".wav")
            metadata = destination.with_suffix(".json")
            if not destination.exists() and not metadata.exists():
                shutil.copy2(audition, destination)
                metadata.write_text(json.dumps(report, ensure_ascii=False, indent=2)+"\n", encoding="utf-8")
                break
            if destination.is_file() and metadata.is_file() and digest(destination) == report["audio_sha256"]:
                if json.loads(metadata.read_text(encoding="utf-8")) == report:
                    break
            take += 1
        saved_audition = destination
    # Inventaire du dossier persistant ; ne pas déclarer une voix validée à l'écoute.
    entries = []
    for voice_folder in sorted(root.iterdir()):
        if not voice_folder.is_dir():
            continue
        manifests = list(voice_folder.glob("*_reference.json"))
        if len(manifests) != 1:
            continue
        info = json.loads(manifests[0].read_text(encoding="utf-8"))
        entries.append({"profile": info.get("profile_id"), "variant": info.get("variant"),
            "reference_sha256": info.get("reference_sha256"), "folder": voice_folder.name,
            "label": info.get("label"), "status": "Créée ; validation d'écoute à conserver séparément",
            "files": [p.name for p in sorted(voice_folder.iterdir()) if p.is_file()]})
    catalogue = {"schema_version": 1, "voices_count": len(entries), "entries": entries}
    temporary = root / (".catalogue-" + uuid.uuid4().hex + ".json")
    temporary.write_text(json.dumps(catalogue, ensure_ascii=False, indent=2)+"\n", encoding="utf-8")
    temporary.replace(root / "catalogue_voix.json")
    # Un fichier ZIP téléchargeable conserve exactement la même arborescence.
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:6]
    archive = root / (prefix + "_" + stamp + ".zip")
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(folder.iterdir()):
            if path.is_file():
                z.write(path, folder.name + "/" + path.name)
    return folder, archive, saved_audition
'''
archive_script = runtime / 'archiver_voix.py'
archive_script.write_text(ARCHIVE_SOURCE, encoding='utf-8')
voice_storage = runpy.run_path(str(archive_script))
def save_current_voice(audition=None):
    folder, archive, saved_audio = voice_storage['save_voice'](VOICES_ROOT, reference_dir, audition)
    print('Voix et JSON enregistrés dans :', folder, flush=True)
    if TELECHARGER_AUTOMATIQUEMENT:
        from google.colab import files
        files.download(str(archive))
    return folder, archive


In [ ]:
from datetime import datetime, timezone
from google.colab import files
from IPython.display import Audio, display

MODE = "Créer une nouvelle voix" #@param ["Créer une nouvelle voix", "Réutiliser une voix sauvegardée"]
PROFIL = "M-30" #@param ['F-petit', 'F-enfant', 'F-ado', 'F-20', 'F-30', 'F-40', 'F-50', 'F-senior', 'M-petit', 'M-enfant', 'M-ado', 'M-20', 'M-30', 'M-40', 'M-50', 'M-senior']
VARIANTE = 1 #@param {type:"integer"}
print(profiles[PROFIL]['label'], '· âge demandé, à vérifier à l’écoute')
reference_dir = Path('casting-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')).resolve()
if MODE == 'Créer une nouvelle voix':
    run_visible([str(qwen_python), '-u', str(qwen_script), '--profile', PROFIL, '--variant', str(VARIANTE), '--output-dir', str(reference_dir)])
    reference_path = reference_dir / 'reference.wav'
else:
    print('Sélectionnez le ZIP voix-fictive.zip conservé lors du précédent casting.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Un seul ZIP de voix est attendu.')
    blob = next(iter(uploaded.values()))
    if len(blob) > 12 * 1024**2:
        raise ValueError('Archive de voix : 12 Mo maximum.')
    archive_input = runtime / ('voix-upload-' + datetime.now(timezone.utc).strftime('%H%M%S%f') + '.zip')
    archive_input.write_bytes(blob)
    reference_path = casting['restore_reference'](archive_input, reference_dir)
voice_info = json.loads((reference_dir / 'voix.json').read_text(encoding='utf-8'))
print('Référence fictive :', voice_info['label'], '· variante', voice_info['variant'])
display(Audio(filename=str(reference_path)))
print('Écoutez ce timbre. Pour comparer une autre voix, changez le profil ou la variante et relancez cette cellule.')

# Sauvegarde et téléchargement immédiatement après la création de la référence.
save_current_voice()


In [ ]:
LANGUE = "ar" #@param ["ar", "fr"]
TEXTE_AR = "مَرْحَبًا بِكُمْ فِي وَرْشَتِنَا. عَلِمَ الطَّالِبُ. عُلِمَ الْخَبَرُ. دَرَسَ الطَّالِبُ. دَرَّسَ الْمُعَلِّمُ." #@param {type:"string"}
TEXTE_FR = "Bonjour. Une nouvelle aventure commence aujourd’hui. Prenons le temps de découvrir cette histoire ensemble." #@param {type:"string"}
text = TEXTE_AR if LANGUE == 'ar' else TEXTE_FR
text_path = runtime / 'texte-casting.txt'
text_path.write_text(text, encoding='utf-8')
result_dir = Path('audition-' + LANGUE + '-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')).resolve()
command = [str(worker_python), '-u', str(chat_script), '--text-file', str(text_path), '--output-dir', str(result_dir), '--language', LANGUE, '--reference', str(reference_path), '--reference-sha256', voice_info['reference_sha256'], '--voice-name', voice_info['label']]
# Réutiliser aussi la révision Chatterbox enregistrée lors de la première audition.
if voice_info.get('chatterbox_revision'):
    command += ['--model-revision', voice_info['chatterbox_revision']]
run_visible(command)
audio_path = result_dir / ('essai-chatterbox-' + LANGUE + '.wav')
report = json.loads(audio_path.with_suffix('.json').read_text(encoding='utf-8'))
voice_info['chatterbox_revision'] = report['model_revision']
voice_info.setdefault('auditions', {})[LANGUE] = {'status':'À écouter', 'audio_sha256': report['audio_sha256'], 'reference_sha256': report['reference_sha256'], 'model_revision': report['model_revision']}
(reference_dir / 'voix.json').write_text(json.dumps(voice_info, ensure_ascii=False, indent=2), encoding='utf-8')
display(Audio(filename=str(audio_path)))
print('Vérifiez naturel, âge perçu, identité, accent, harakat et phrases complètes. Pour l’autre langue, changez LANGUE et relancez seulement cette cellule.')

# Chaque réplique et son JSON rejoignent automatiquement le même dossier.
save_current_voice(audio_path)


In [ ]:
# Sauvegarde manuelle supplémentaire si nécessaire : un seul ZIP, avec WAV + JSON nommés.
audition_to_save = None
if 'audio_path' in globals() and audio_path.is_file():
    current_report = json.loads(audio_path.with_suffix('.json').read_text(encoding='utf-8'))
    if current_report.get('reference_sha256') == voice_info['reference_sha256']:
        audition_to_save = audio_path
saved_folder, saved_zip = save_current_voice(audition_to_save)
print('Le ZIP contient le dossier complet. Le sous-fichier voix-fictive.zip permet de réutiliser la voix dans ce notebook.')
